# Fantasy Football Predictions

Phase 1 (now)
└── Local pipeline → XGBoost points model → Streamlit tab

Phase 2
└── Cloud storage (S3/GCS) for data + model artifacts

Phase 3
└── Databricks pipeline with Delta Lake + Spark

Phase 4
└── K-Means clustering → player archetypes + breakout signals
    └── Learn: unsupervised learning, feature scaling, cluster evaluation

Phase 5
└── Neural network (LSTM) as alternative points model
    └── Learn: deep learning, time series, PyTorch

Phase 6
└── LLM explanation layer
    └── Learn: fine tuning, RAG

Phase 7
└── MLflow + Evidently monitoring
    └── Learn: ML lifecycle, drift detection

Phase 8 (optional)
└── Palantir Foundry ontology

In [67]:
import nflreadpy as nfl
import pandas as pd

# nflreadpy returns Polars DataFrames — we'll convert to pandas right after loading
# Seasons to pull
SEASONS = list(range(2020, 2026))

# Scoring format
SCORING = "half_ppr"  # change to "std" or "ppr" if needed

SCORING_WEIGHTS = {
    "half_ppr": {
        "pass_yd": 0.04, "pass_td": 4,  "pass_int": -2,
        "rush_yd": 0.1,  "rush_td": 6,
        "rec":     0.5,  "rec_yd": 0.1, "rec_td": 6,
        "fumble":  -2
    }
}

print("✅ Config ready")

✅ Config ready


In [68]:
player_stats = nfl.load_player_stats(SEASONS).to_pandas()

# Regular season only, skill positions only
player_stats = player_stats[player_stats["season_type"] == "REG"]
player_stats = player_stats[player_stats["position"].isin(["QB", "RB", "WR", "TE"])]

print(player_stats.shape)
print(player_stats.columns.tolist())

(34906, 115)
['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'week', 'season_type', 'team', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share', 'wopr', 'special_teams_tds', 'def_tackles_solo', 'def_tackles_with_assist', 'def_tackle_assists', 'def_tack

In [69]:
ff_opp = nfl.load_ff_opportunity(SEASONS).to_pandas()

print(ff_opp.shape)
print(ff_opp.columns.tolist())

(36063, 159)
['season', 'posteam', 'week', 'game_id', 'player_id', 'full_name', 'position', 'pass_attempt', 'rec_attempt', 'rush_attempt', 'pass_air_yards', 'rec_air_yards', 'pass_completions', 'receptions', 'pass_completions_exp', 'receptions_exp', 'pass_yards_gained', 'rec_yards_gained', 'rush_yards_gained', 'pass_yards_gained_exp', 'rec_yards_gained_exp', 'rush_yards_gained_exp', 'pass_touchdown', 'rec_touchdown', 'rush_touchdown', 'pass_touchdown_exp', 'rec_touchdown_exp', 'rush_touchdown_exp', 'pass_two_point_conv', 'rec_two_point_conv', 'rush_two_point_conv', 'pass_two_point_conv_exp', 'rec_two_point_conv_exp', 'rush_two_point_conv_exp', 'pass_first_down', 'rec_first_down', 'rush_first_down', 'pass_first_down_exp', 'rec_first_down_exp', 'rush_first_down_exp', 'pass_interception', 'rec_interception', 'pass_interception_exp', 'rec_interception_exp', 'rec_fumble_lost', 'rush_fumble_lost', 'pass_fantasy_points_exp', 'rec_fantasy_points_exp', 'rush_fantasy_points_exp', 'pass_fantasy_p

In [70]:
schedules = nfl.load_schedules(SEASONS).to_pandas()

# Regular season only
schedules = schedules[schedules["game_type"] == "REG"]

print(schedules.shape)
print(schedules.columns.tolist())

(1615, 46)
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']


In [71]:
ps_cols = [
    # identifiers
    "player_id", "player_display_name", "position", "team",
    "opponent_team", "season", "week", "game_id",
    # target variable (already calculated for us)
    "fantasy_points", "fantasy_points_ppr",
    # passing
    "completions", "attempts", "passing_yards", "passing_tds",
    "passing_interceptions", "passing_air_yards", "passing_epa",
    # rushing
    "carries", "rushing_yards", "rushing_tds",
    "rushing_fumbles_lost", "rushing_epa",
    # receiving
    "receptions", "targets", "receiving_yards", "receiving_tds",
    "receiving_fumbles_lost", "receiving_air_yards",
    "receiving_yards_after_catch", "receiving_epa",
    # usage/opportunity
    "target_share", "air_yards_share", "wopr", "racr",
]

player_stats_clean = player_stats[ps_cols].copy()

print(player_stats_clean.shape)
player_stats_clean.tail()

(34906, 34)


,player_id,player_display_name,position,team,opponent_team,season,week,game_id,fantasy_points,fantasy_points_ppr,...,receiving_yards,receiving_tds,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_epa,target_share,air_yards_share,wopr,racr
111553,00-0040730,RJ Harvey,RB,DEN,LAC,2025,18,2025_18_LAC_DEN,3.30,4.30,...,5,0,0,-5,8,-3.856058,0.173913,-0.192308,0.126254,-1.000000
111555,00-0040734,TreVeyon Henderson,RB,NE,MIA,2025,18,2025_18_MIA_NE,17.30,17.30,...,0,0,0,0,0,NaN,0.000000,0.000000,0.000000,NaN
111556,00-0040735,Luther Burden III,WR,CHI,DET,2025,18,2025_18_DET_CHI,4.50,7.50,...,35,0,0,12,27,2.777708,0.137931,0.046154,0.239204,2.916667
111558,00-0040743,Tyler Shough,QB,NO,ATL,2025,18,2025_18_NO_ATL,21.76,21.76,...,0,0,0,0,0,NaN,0.000000,0.000000,0.000000,NaN
111563,00-0040782,Isaiah Bond,WR,CLE,CIN,2025,18,2025_18_CLE_CIN,1.70,2.70,...,13,0,0,48,0,0.556947,0.095238,0.244898,0.314286,0.270833


In [72]:
# ff_opp gives us expected points and the diff (actual - expected)
# total_fantasy_points_diff = actual minus expected = our over/underperformance signal
ffo_cols = [
    "player_id", "season", "week", "game_id",
    "total_fantasy_points",        # actual (their version)
    "total_fantasy_points_exp",    # expected based on opportunities
    "total_fantasy_points_diff",   # actual minus expected (key feature)
    "rec_attempt",                 # targets from their model
    "rush_attempt",
    "rec_yards_gained_exp",
    "rush_yards_gained_exp",
    "rec_touchdown_exp",
    "rush_touchdown_exp",
]

ffo_clean = ff_opp[ffo_cols].copy()
ffo_clean = ffo_clean.rename(columns={
    "total_fantasy_points":      "ffo_actual_pts",
    "total_fantasy_points_exp":  "ffo_expected_pts",
    "total_fantasy_points_diff": "ffo_pts_diff",   # positive = outperforming, negative = underperforming
})

print(ffo_clean.shape)
ffo_clean.head()

(36063, 13)


,player_id,season,week,game_id,ffo_actual_pts,ffo_expected_pts,ffo_pts_diff,rec_attempt,rush_attempt,rec_yards_gained_exp,rush_yards_gained_exp,rec_touchdown_exp,rush_touchdown_exp
0,00-0031345,2020,1.0,2020_01_ARI_SF,19.26,17.66,1.60,0.0,1.0,0.00,7.04,0.00,0.00
1,00-0033288,2020,1.0,2020_01_ARI_SF,9.30,10.28,-0.98,5.0,1.0,38.61,5.87,0.23,0.09
2,00-0035228,2020,1.0,2020_01_ARI_SF,26.30,19.53,6.77,0.0,13.0,0.00,75.43,0.00,0.07
3,00-0030564,2020,1.0,2020_01_ARI_SF,29.10,24.03,5.07,16.0,0.0,118.96,0.00,0.10,0.00
4,00-0022921,2020,1.0,2020_01_ARI_SF,7.40,7.22,0.18,5.0,0.0,32.60,0.00,0.01,0.00


In [73]:
# We need implied team total for each team each week
# Formula: home implied = (total_line - spread_line) / 2
#          away implied = (total_line + spread_line) / 2

home = schedules[["season", "week", "home_team", "away_team", "total_line", "spread_line"]].copy()
home = home.rename(columns={"home_team": "team", "away_team": "opponent"})
home["implied_team_total"] = (home["total_line"] - home["spread_line"]) / 2
home["is_home"] = 1

away = schedules[["season", "week", "away_team", "home_team", "total_line", "spread_line"]].copy()
away = away.rename(columns={"away_team": "team", "home_team": "opponent"})
away["implied_team_total"] = (home["total_line"] + home["spread_line"]) / 2
away["is_home"] = 0

vegas = pd.concat([home, away], ignore_index=True)[
    ["season", "week", "team", "implied_team_total", "is_home"]
]

print(vegas.shape)
vegas.head()

(3230, 5)


,season,week,team,implied_team_total,is_home
0,2020,1,KC,22.00,1
1,2020,1,ATL,24.25,1
2,2020,1,BAL,20.00,1
3,2020,1,BUF,16.50,1
4,2020,1,CAR,25.50,1


In [74]:
# Fix dtypes so all key columns match before joining
for col in ["season", "week"]:
    player_stats_clean[col] = player_stats_clean[col].astype(int)
    ffo_clean[col] = ffo_clean[col].astype(int)
    vegas[col] = vegas[col].astype(int)

# Make sure game_id is string in both
player_stats_clean["game_id"] = player_stats_clean["game_id"].astype(str)
ffo_clean["game_id"] = ffo_clean["game_id"].astype(str)

# Make sure player_id is string in both
player_stats_clean["player_id"] = player_stats_clean["player_id"].astype(str)
ffo_clean["player_id"] = ffo_clean["player_id"].astype(str)

# Now join
df = player_stats_clean.copy()
df = df.merge(ffo_clean, on=["player_id", "season", "week", "game_id"], how="left")
df = df.merge(vegas, left_on=["team", "season", "week"],
                     right_on=["team", "season", "week"], how="left")

df = df.sort_values(["player_id", "season", "week"]).reset_index(drop=True)

print(df.shape)
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head()

(34906, 45)
passing_epa              30939
rushing_epa              21640
receiving_epa             9370
racr                      9547
ffo_actual_pts           19016
ffo_expected_pts         19016
ffo_pts_diff             19016
rec_attempt              19016
rush_attempt             19016
rec_yards_gained_exp     19016
rush_yards_gained_exp    19016
rec_touchdown_exp        19016
rush_touchdown_exp       19016
dtype: int64


,player_id,player_display_name,position,team,opponent_team,season,week,game_id,fantasy_points,fantasy_points_ppr,...,ffo_expected_pts,ffo_pts_diff,rec_attempt,rush_attempt,rec_yards_gained_exp,rush_yards_gained_exp,rec_touchdown_exp,rush_touchdown_exp,implied_team_total,is_home
0,00-0019596,Tom Brady,QB,TB,NO,2020,1,None,20.46,20.46,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,26.25,0
1,00-0019596,Tom Brady,QB,TB,CAR,2020,2,None,8.68,8.68,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19.75,1
2,00-0019596,Tom Brady,QB,TB,DEN,2020,3,None,23.88,23.88,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.25,0
3,00-0019596,Tom Brady,QB,TB,LAC,2020,4,None,32.46,32.46,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.50,1
4,00-0019596,Tom Brady,QB,TB,CHI,2020,5,None,14.12,14.12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,20.25,0


In [75]:
# Show all Justin Jefferson rows, no truncation
pd.set_option('display.max_rows', None)

sample = df[df["player_display_name"] == "Justin Jefferson"]
print(sample[["season", "week", "team", "targets", "receiving_yards",
              "fantasy_points", "ffo_expected_pts", "ffo_pts_diff",
              "implied_team_total", "is_home"]].to_string())

pd.reset_option('display.max_rows')

       season  week team  targets  receiving_yards  fantasy_points  ffo_expected_pts  ffo_pts_diff  implied_team_total  is_home
21018    2020     1  MIN        3               26            2.60               NaN           NaN               22.00        1
21019    2020     2  MIN        3               44            4.40               NaN           NaN               26.25        0
21020    2020     3  MIN        9              175           23.50               NaN           NaN               26.25        1
21021    2020     4  MIN        5              103           10.30               NaN           NaN               28.00        0
21022    2020     5  MIN        5               23            2.30               NaN           NaN               30.25        0
21023    2020     6  MIN       11              166           30.60               NaN           NaN               24.75        1
21024    2020     8  MIN        4               26            2.60               NaN           NaN      

In [76]:
# Check how many unique players are in each table
print("Unique players in player_stats:", player_stats_clean["player_id"].nunique())
print("Unique players in ffo:", ffo_clean["player_id"].nunique())

# Check overlap
ps_ids = set(player_stats_clean["player_id"].unique())
ffo_ids = set(ffo_clean["player_id"].unique())
print("Players in both:", len(ps_ids & ffo_ids))
print("Players only in player_stats (no ffo coverage):", len(ps_ids - ffo_ids))

Unique players in player_stats: 1240
Unique players in ffo: 1358
Players in both: 1190
Players only in player_stats (no ffo coverage): 50


In [77]:
# ffo coverage is partial — fill nulls with 0 for players with no opportunity data
# These are mostly low-usage players where expected pts ~ 0 anyway
ffo_fill_cols = [
    "ffo_actual_pts", "ffo_expected_pts", "ffo_pts_diff",
    "rec_attempt", "rush_attempt",
    "rec_yards_gained_exp", "rush_yards_gained_exp",
    "rec_touchdown_exp", "rush_touchdown_exp"
]

df[ffo_fill_cols] = df[ffo_fill_cols].fillna(0)

# Verify no more nulls in these columns
print("Remaining nulls:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Remaining nulls:
passing_epa      30939
rushing_epa      21640
receiving_epa     9370
racr              9547
dtype: int64


In [78]:
# Save the joined dataset — this becomes the input to features.py
df.to_csv(r"C:\Users\josep\iCloudDrive\Projects\BettingEdgeContinued\fantasy\raw_dataset.csv", index=False)
print(f"✅ Saved — shape: {df.shape}")

✅ Saved — shape: (34906, 45)
